In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Move the models from drive to local models/ folder
!mkdir -p models
!cp -r "/content/drive/MyDrive/CS521_HW1_Models/." models/

## Results from Problem 2 (single- and multi-norm robustness)

From the Problem 2 solution, we already have the following robust accuracies on the CIFAR-10 test set (PGD, 10 steps, step size = ε/4):

| Model | ℓ∞ robust acc (ε = 8/255) | ℓ2 robust acc (ε = 0.75) | Union (ℓ∞ and ℓ2) acc |
|---|---|---|---|
| pretr_Linf | 51.52% | 49.35% | 47.47% |
| pretr_L2 | 31.41% | 54.89% | 31.48% |
| pretr_RAMP | 50.04% | 59.87% | 50.01% |

Below, we evaluate the same three models on CIFAR-10-UA to see how this robustness carries over to unforeseen attacks.


# CIFAR-10-UA evaluation (using the official repo directly)

Runs the repo's own evaluation script (`main.py`) on our three models against the 8 core attacks. Settings (epsilons, step sizes, number of steps, attack hyperparameters) are the repo's CIFAR-10 defaults, the same ones its `eval.sh` uses.

The repo already includes this PreActResNet-18 (`architecture=preactresnet18`), so our weights load directly. Two one-line patches are needed:
1. The repo builds the model with ReLU, but our models use `activation='softplus1'`.
2. `parse_args.py` errors out because it uses `config` without importing it.
Added the import statement.

In [3]:
%cd /content
# !rm -rf /content/adversarial-corruptions
!git clone -q https://github.com/centerforaisafety/adversarial-corruptions.git
!pip install -q jsonlines timm
%cd /content/adversarial-corruptions

# Our models use softplus activations, but the repo builds this model with ReLU
!sed -i 's/model = PreActResNet18(n_cls=10)$/model = PreActResNet18(n_cls=10, activation="softplus1")/' models/cifar10/preactresnet18.py
# parse_args.py uses `config` without importing it
!sed -i '1i import config' parse_args.py

/content
/content/adversarial-corruptions


In [4]:
import subprocess
from tqdm.auto import tqdm

MODELS = ["pretr_Linf", "pretr_L2", "pretr_RAMP"]
LEVEL = "medium"
NUM_BATCHES = 1   # batches of 1*BATCH_SIZE images; 1 = 1,000 test images, -1 = full 10,000
BATCH_SIZE = 1000

# (step size, number of steps) for each attack on CIFAR-10, from the repo's models.sh
ATTACKS = {
    "gabor":        (0.0025, 80),
    "snow":         (0.2, 20),
    "pixel":        (1.0, 60),
    "jpeg":         (0.0024, 50),
    "kaleidoscope": (0.005, 30),
    "glitch":       (0.0025, 60),
    "wood":         (0.000625, 70),
    "elastic":      (0.006, 30),
}

for model in MODELS:
    print(f"Model: {model}")
    accs = []
    for attack, (step_size, num_steps) in tqdm(ATTACKS.items(), desc=model):
        cmd = ["python", "main.py",
               "--dataset", "cifar10", "--architecture", "preactresnet18",
               "--weights", f"/content/models/{model}.pth",
               "--attack", attack, "--epsilon", LEVEL,
               "--step_size", str(step_size), "--num_steps", str(num_steps),
               "--attack_config", "attacks/attack_params_cifar10_default.conf",
               "--eval_batch_size", str(BATCH_SIZE), "--num_batches", str(NUM_BATCHES),
               "--seed", "123"]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        acc = float(result.stdout.split()[-1])  # main.py prints the accuracy last
        accs.append(acc)
        tqdm.write(f"  {attack}: {acc:.4f}")
    print(f"  UA2 ({LEVEL}): {sum(accs) / len(accs):.4f}\n")

Model: pretr_Linf


pretr_Linf:   0%|          | 0/8 [00:00<?, ?it/s]

  gabor: 0.5370
  snow: 0.3340
  pixel: 0.1550
  jpeg: 0.4870
  kaleidoscope: 0.3390
  glitch: 0.6260
  wood: 0.3310
  elastic: 0.4700
  UA2 (medium): 0.4099

Model: pretr_L2


pretr_L2:   0%|          | 0/8 [00:00<?, ?it/s]

  gabor: 0.5560
  snow: 0.1860
  pixel: 0.1450
  jpeg: 0.5980
  kaleidoscope: 0.2370
  glitch: 0.6220
  wood: 0.3790
  elastic: 0.3900
  UA2 (medium): 0.3891

Model: pretr_RAMP


pretr_RAMP:   0%|          | 0/8 [00:00<?, ?it/s]

  gabor: 0.5690
  snow: 0.3240
  pixel: 0.2450
  jpeg: 0.6080
  kaleidoscope: 0.3450
  glitch: 0.6440
  wood: 0.5040
  elastic: 0.4820
  UA2 (medium): 0.4651

